# 02 — Model Training
Random Forest · SVM · Gradient Boosting · Stacked Ensemble


In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, StackingClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, cohen_kappa_score, accuracy_score, f1_score

FEATURES = ['Endurance','Strength','Speed','Flexibility','Cognitive Ability','Reflex']

df = pd.read_csv('../data/combined_sports_data.csv').drop_duplicates().dropna()
for c in FEATURES:
    df[c] = df[c].clip(1, 10)

le = LabelEncoder()
y  = le.fit_transform(df['Sport'])
X  = df[FEATURES].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Sports: {list(le.classes_)}')

In [ ]:
# ── Cross-validation comparison ──
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models_cv = {
    'Random Forest':     (RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42), X),
    'SVM (RBF)':         (SVC(kernel='rbf', C=10, gamma='scale', class_weight='balanced', probability=True, random_state=42), scaler.fit_transform(X)),
    'Gradient Boosting': (GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42), X),
}

print(f'{"Model":<22s}  {"CV Accuracy":>12s}  {"CV Macro F1":>12s}')
print('─' * 50)
for name, (model, Xm) in models_cv.items():
    acc = cross_val_score(model, Xm, y, cv=cv, scoring='accuracy', n_jobs=-1)
    f1  = cross_val_score(model, Xm, y, cv=cv, scoring='f1_macro',  n_jobs=-1)
    print(f'{name:<22s}  {acc.mean():.4f}±{acc.std():.4f}  {f1.mean():.4f}±{f1.std():.4f}')

In [ ]:
# ── Train final models ──
rf  = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1)
svm = SVC(kernel='rbf', C=10, gamma='scale', class_weight='balanced', probability=True, random_state=42)
gb  = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42)

rf.fit(X_train,    y_train)
svm.fit(X_train_sc, y_train)
gb.fit(X_train,    y_train)

# Stacked ensemble
ensemble = StackingClassifier(
    estimators=[
        ('rf',  Pipeline([('rf',  RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42))])),
        ('svm', Pipeline([('sc',  StandardScaler()), ('svm', SVC(kernel='rbf', C=10, gamma='scale', class_weight='balanced', probability=True, random_state=42))])),
        ('gb',  Pipeline([('gb',  GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42))])),
    ],
    final_estimator=LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    cv=5, stack_method='predict_proba', n_jobs=-1
)
ensemble.fit(X_train, y_train)
print('All models trained.')

In [ ]:
# ── Test-set results ──
for name, model, Xt in [
    ('Random Forest',    rf,       X_test),
    ('SVM',              svm,      X_test_sc),
    ('Gradient Boost',   gb,       X_test),
    ('Stacked Ensemble', ensemble, X_test),
]:
    preds = model.predict(Xt)
    print(f'\n── {name} ──')
    print(f'  Accuracy    : {accuracy_score(y_test, preds):.4f}')
    print(f'  Macro F1    : {f1_score(y_test, preds, average="macro"):.4f}')
    print(f'  Cohen Kappa : {cohen_kappa_score(y_test, preds):.4f}')

In [ ]:
# ── Top-3 hit rate (clinically meaningful metric) ──
proba  = ensemble.predict_proba(X_test)
top3   = np.argsort(proba, axis=1)[:, -3:]
hits   = sum(y_test[i] in top3[i] for i in range(len(y_test)))
print(f'Top-3 hit rate: {hits/len(y_test):.4f}  ({hits}/{len(y_test)} samples)')
print('(Fraction where the correct sport is in model\'s top-3 recommendations)')

In [ ]:
# ── Save models ──
import os
os.makedirs('../models', exist_ok=True)
joblib.dump(rf,       '../models/random_forest_model.pkl')
joblib.dump(svm,      '../models/svm_model.pkl')
joblib.dump(gb,       '../models/gradient_boosting_model.pkl')
joblib.dump(ensemble, '../models/ensemble_model.pkl')
joblib.dump(scaler,   '../models/scaler.pkl')
joblib.dump(le,       '../models/label_encoder.pkl')
print('Models saved to ../models/')